# Generating Dataset By code 

In [12]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

patient_id = [f"P{i:04d}" for i in range(1, n + 1)]
age = np.clip(rng.normal(50, 14, n), 18, 90).round().astype(int)
gender = rng.choice(["Male", "Female"], n, p=[0.52, 0.48])
region = rng.choice(["North", "South", "East", "West"], n, p=[0.30, 0.25, 0.25, 0.20])

bmi = np.clip(24 + 0.05 * (age - 50) + rng.normal(0, 3.5, n), 16, 42)
blood_pressure = 115 + 0.35 * (age - 50) + 0.8 * (bmi - 25) + rng.normal(0, 9, n)
cholesterol = 190 + 0.6 * (age - 50) + 1.2 * (bmi - 25) + rng.normal(0, 28, n)
glucose = 92 + 0.2 * (age - 50) + 1.0 * (bmi - 25) + rng.normal(0, 12, n)

z = (0.03 * (age - 50) + 0.08 * (bmi - 25) + 0.03 * (blood_pressure - 115)
     + 0.012 * (cholesterol - 190) + 0.03 * (glucose - 92)
     + 0.3 * (gender == "Male") - 0.9)
prob = 1 / (1 + np.exp(-z))
disease_risk = (rng.random(n) < prob).astype(int)

df = pd.DataFrame({
    "patient_id": patient_id, "age": age, "gender": gender, "region": region,
    "bmi": bmi, "blood_pressure": blood_pressure,
    "cholesterol": cholesterol, "glucose": glucose,
    "disease_risk": disease_risk,
})

def inject_outliers(col, low_range, high_range, frac=0.03, high_share=0.7):
    idx = rng.choice(n, int(n * frac), replace=False)
    for i in idx:
        if low_range is None or rng.random() < high_share:
            df.loc[i, col] = rng.uniform(*high_range)
        else:
            df.loc[i, col] = rng.uniform(*low_range)
    return set(idx)

out_bmi  = inject_outliers("bmi", (10, 13), (50, 65))
out_bp   = inject_outliers("blood_pressure", None, (200, 260))
out_chol = inject_outliers("cholesterol", (70, 100), (350, 450))
out_glu  = inject_outliers("glucose", None, (250, 400))

df["bmi"] = df["bmi"].round(1)
df["blood_pressure"] = df["blood_pressure"].round(1)
df["cholesterol"] = df["cholesterol"].round(1)
df["glucose"] = df["glucose"].round(1)

missing_frac = {"age": 0.05, "gender": 0.06, "region": 0.07,
                "bmi": 0.08, "cholesterol": 0.10, "glucose": 0.09}
outlier_rows = {"bmi": out_bmi, "cholesterol": out_chol, "glucose": out_glu}

df["age"] = df["age"].astype("Int64")
for col, frac in missing_frac.items():
    pool = [i for i in range(n) if i not in outlier_rows.get(col, set())]
    idx = rng.choice(pool, int(n * frac), replace=False)
    df.loc[idx, col] = pd.NA

df.to_csv("patient_health_records.csv", index=False)
print(df.shape)
print((df.isna().mean() * 100).round(1))
print(df["disease_risk"].value_counts())

(1000, 9)
patient_id         0.0
age                5.0
gender             6.0
region             7.0
bmi                8.0
blood_pressure     0.0
cholesterol       10.0
glucose            9.0
disease_risk       0.0
dtype: float64
disease_risk
0    683
1    317
Name: count, dtype: int64


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

# DataSet

In [4]:
df

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,P0001,54,Female,East,21.8,104.5,157.5,88.6,0
1,P0002,35,Male,<NA>,30.4,108.5,178.1,95.8,0
2,P0003,61,Male,North,24.4,109.1,352.3,112.0,1
3,P0004,63,<NA>,East,28.2,108.1,196.7,364.6,0
4,P0005,23,Male,East,NaN,105.5,NaN,86.2,0
...,...,...,...,...,...,...,...,...,...
995,P0996,55,Male,East,26.1,119.9,238.9,91.4,1
996,P0997,61,Male,<NA>,23.0,112.8,NaN,97.6,1
997,P0998,52,Male,North,23.3,108.5,242.5,108.2,1
998,P0999,<NA>,Male,North,21.3,113.9,NaN,92.3,1


In [5]:
df.shape

(1000, 9)

In [6]:
df.isnull().sum()

patient_id          0
age                50
gender             60
region             70
bmi                80
blood_pressure      0
cholesterol       100
glucose            90
disease_risk        0
dtype: int64

# Task A : Handling Missing Values

In [7]:
# Percentage of Missing values in columns 
df.isnull().sum()/len(df)*100

patient_id         0.0
age                5.0
gender             6.0
region             7.0
bmi                8.0
blood_pressure     0.0
cholesterol       10.0
glucose            9.0
disease_risk       0.0
dtype: float64

#Simple Imputer to fill BMI column

In [17]:
df["bmi"].unique()

array([21.8, 30.4, 24.4, 28.2, 24.1, 23. , 22.1, 24.2, 25.1, 20.7, 22.4,
       29.8, 22.7, 25.2, 18.7, 24.7, 29.4, 24.3, 29.7, 26.1, 23.5, 32.7,
       22.6, 21.6, 20.1, 33.7, 26.6, 22.5, 27.8, 26.9, 31.7, 17.5, 32.4,
       25. , 27.6, 27.9, 25.8, 16. , 28.9, 21.1, 24.5, 23.4, 27.3, 18. ,
       20.8, 26.5, 27.5, 26.3, 22.8, 20.3, 23.1, 28.8, 23.7, 21.4, 22. ,
       24.8, 25.5, 26.7, 28.7, 27.2, 19.9, 21.2, 20.4, 55. , 28.1, 23.6,
       22.3, 32.2, 18.4, 26. , 18.3, 19.7, 20.2, 27.7, 19.8, 29.5, 19.6,
       28. , 21.5, 30.1, 50.5, 25.6, 18.8, 27.4, 18.5, 24.6, 21. , 20. ,
       28.4, 26.2, 30.6, 23.3, 27.1, 25.7, 34.5, 21.3, 25.3, 26.8, 11.8,
       25.9, 17.2, 64.4, 22.2, 52.1, 19.1, 22.9, 19.3, 63.9, 24.9, 19.2,
       26.4, 64.6, 62.2, 56.2, 32.1, 28.3, 17.8, 29.6, 20.9, 21.7, 23.9,
       21.9, 18.2, 28.6, 58.6, 23.8, 25.4, 24. , 31.2, 27. , 35. , 18.9,
       23.2, 31.6, 20.6, 20.5, 16.6, 19. , 30.7, 17.4, 17.9, 16.7, 33.3,
       30. , 16.9, 30.5, 63.8, 53. , 59.1, 29. , 29

In [17]:
median_imputer = SimpleImputer(strategy="median")
df["bmi"] = median_imputer.fit_transform(df[["bmi"]]).ravel()

#Simple Imputer to fill Region column

In [18]:
df["region"] = df["region"].where(df["region"].notna(), np.nan)
region_imputer = SimpleImputer(strategy="most_frequent")
df["region"] = region_imputer.fit_transform(df[["region"]]).ravel()

#filling Gender column with mode

In [19]:
df["region"] = df["region"].where(df["region"].notna(), np.nan)
region_imputer = SimpleImputer(strategy="most_frequent")
df["region"] = region_imputer.fit_transform(df[["region"]]).ravel()

#KNN Imputer

In [21]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

df_original = pd.read_csv("patient_health_records.csv")
num_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

df_knn = df_original.copy()

scaler = StandardScaler()                          # KNN uses distance, so scale first
scaled = scaler.fit_transform(df_knn[num_cols])

knn_filled = KNNImputer(n_neighbors=5).fit_transform(scaled)
df_knn[num_cols] = scaler.inverse_transform(knn_filled)   # convert back to real units
df_knn["age"] = df_knn["age"].round().astype(int)

# KNNImputer only works on numbers, so handle the categorical columns separately
df_knn["gender"] = df_knn["gender"].fillna(df_knn["gender"].mode()[0])
df_knn["region"] = df_knn["region"].fillna(df_knn["region"].mode()[0])

print("Missing left:", df_knn.isnull().sum().sum())
df_knn[num_cols].describe().round(2)

Missing left: 0


,age,bmi,blood_pressure,cholesterol,glucose
count,1000.00,1000.00,1000.00,1000.00,1000.00
mean,49.58,24.96,117.50,194.16,99.76
std,13.51,6.56,23.06,43.59,44.64
min,18.00,11.60,72.70,46.20,50.30
25%,41.00,21.80,107.38,170.60,83.60
50%,50.00,24.20,114.50,192.20,91.40
75%,58.00,26.50,121.50,211.42,101.40
max,90.00,64.70,258.50,440.40,395.00


#Mice Algorithm

In [22]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

df_mice = df_original.copy()

mice_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
df_mice[num_cols] = mice_imputer.fit_transform(df_mice[num_cols])
df_mice["age"] = df_mice["age"].round().astype(int)

df_mice["gender"] = df_mice["gender"].fillna(df_mice["gender"].mode()[0])
df_mice["region"] = df_mice["region"].fillna(df_mice["region"].mode()[0])

print("Missing left:", df_mice.isnull().sum().sum())
df_mice[num_cols].describe().round(2)

Missing left: 0


,age,bmi,blood_pressure,cholesterol,glucose
count,1000.00,1000.00,1000.00,1000.00,1000.00
mean,49.49,24.95,117.50,193.91,99.19
std,13.39,6.51,23.06,42.93,43.93
min,18.00,11.60,72.70,46.20,50.30
25%,41.00,21.90,107.38,171.15,83.80
50%,50.00,24.40,114.50,192.54,92.90
75%,58.00,26.30,121.50,209.80,99.50
max,90.00,64.70,258.50,440.40,395.00


# Part B : Handling Outliers

In [23]:
df_original = pd.read_csv("patient_health_records.csv")
num_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

df_clean = df_original.copy()
mice_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
df_clean[num_cols] = mice_imputer.fit_transform(df_clean[num_cols])
df_clean["age"] = df_clean["age"].round().astype(int)
df_clean["gender"] = df_clean["gender"].fillna(df_clean["gender"].mode()[0])
df_clean["region"] = df_clean["region"].fillna(df_clean["region"].mode()[0])

print("Missing values left:", df_clean.isnull().sum().sum())
print("Shape:", df_clean.shape)

Missing values left: 0
Shape: (1000, 9)


#Z-score method — cholesterol & glucose

In [24]:
def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold

chol_outliers = zscore_outliers(df_clean["cholesterol"])
glucose_outliers = zscore_outliers(df_clean["glucose"])
print("Cholesterol outliers:", chol_outliers.sum())
print("Glucose outliers:", glucose_outliers.sum())

z_outlier_mask = chol_outliers | glucose_outliers
df_zscore_removed = df_clean[~z_outlier_mask].reset_index(drop=True)
print("Shape before:", df_clean.shape, "→ after removal:", df_zscore_removed.shape)

Cholesterol outliers: 23
Glucose outliers: 30
Shape before: (1000, 9) → after removal: (948, 9)


#IQR method — BMI

In [25]:
Q1 = df_clean["bmi"].quantile(0.25)
Q3 = df_clean["bmi"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"Normal BMI range: {lower_bound:.1f} to {upper_bound:.1f}")

bmi_outliers = (df_clean["bmi"] < lower_bound) | (df_clean["bmi"] > upper_bound)
print("BMI outliers:", bmi_outliers.sum())

df_iqr_removed = df_clean[~bmi_outliers].reset_index(drop=True)
print("Shape after IQR removal:", df_iqr_removed.shape)

Normal BMI range: 15.3 to 32.9
BMI outliers: 38
Shape after IQR removal: (962, 9)


#Percentile method — cap at 1st/99th percentile

In [26]:
df_percentile = df_clean.copy()
cols_to_cap = ["bmi", "blood_pressure", "cholesterol", "glucose"]

for col in cols_to_cap:
    lower_p = df_clean[col].quantile(0.01)
    upper_p = df_clean[col].quantile(0.99)
    df_percentile[col] = df_clean[col].clip(lower=lower_p, upper=upper_p)

df_percentile[cols_to_cap].describe().round(2)

,bmi,blood_pressure,cholesterol,glucose
count,1000.00,1000.00,1000.00,1000.00
mean,24.95,117.46,193.94,99.09
std,6.37,22.43,41.25,42.95
min,16.20,88.39,104.51,60.49
25%,21.90,107.38,171.15,83.80
50%,24.40,114.50,192.54,92.90
75%,26.30,121.50,209.80,99.50
max,61.80,243.60,405.47,367.61


#Winsorization

In [27]:
from scipy.stats.mstats import winsorize

df_winsor = df_clean.copy()
for col in cols_to_cap:
    df_winsor[col] = winsorize(df_clean[col].values, limits=[0.01, 0.01])

df_winsor[cols_to_cap].describe().round(2)

,bmi,blood_pressure,cholesterol,glucose
count,1000.00,1000.00,1000.00,1000.00
mean,24.95,117.46,193.94,99.09
std,6.37,22.43,41.25,42.95
min,16.20,88.40,104.60,60.50
25%,21.90,107.38,171.15,83.80
50%,24.40,114.50,192.54,92.90
75%,26.30,121.50,209.80,99.50
max,61.80,243.60,405.40,367.60


#Compare shape and summary — before vs after

In [28]:
print("Original shape:            ", df_clean.shape)
print("After Z-score removal:     ", df_zscore_removed.shape)
print("After IQR removal:         ", df_iqr_removed.shape)
print("After Percentile capping:  ", df_percentile.shape, "(rows unchanged, values capped)")
print("After Winsorization:       ", df_winsor.shape, "(rows unchanged, values capped)")

comparison = pd.DataFrame({
    "Before": df_clean[cols_to_cap].agg(["mean", "std", "min", "max"]).stack(),
    "Percentile": df_percentile[cols_to_cap].agg(["mean", "std", "min", "max"]).stack(),
    "Winsorized": df_winsor[cols_to_cap].agg(["mean", "std", "min", "max"]).stack(),
}).round(2)
comparison

Original shape:             (1000, 9)
After Z-score removal:      (948, 9)
After IQR removal:          (962, 9)
After Percentile capping:   (1000, 9) (rows unchanged, values capped)
After Winsorization:        (1000, 9) (rows unchanged, values capped)


Before  Percentile  Winsorized
mean bmi              24.95       24.95       24.95
     blood_pressure  117.50      117.46      117.46
     cholesterol     193.91      193.94      193.94
     glucose          99.19       99.09       99.09
std  bmi               6.51        6.37        6.37
     blood_pressure   23.06       22.43       22.43
     cholesterol      42.93       41.25       41.25
     glucose          43.93       42.95       42.95
min  bmi              11.60       16.20       16.20
     blood_pressure   72.70       88.39       88.40
     cholesterol      46.20      104.51      104.60
     glucose          50.30       60.49       60.50
max  bmi              64.70       61.80       61.80
     blood_pressure  258.50      243.60      243.60
     cholesterol     440.40      405.47      405.40
     glucose         395.00      367.61      367.60